# 1.2 Comprehensions, Generators & Decorators> These are Python's superpowers — no direct Java equivalents.---

## List Comprehensions☕ **Java parallel:** Like `stream().filter().map().collect()` but in one line.

In [ ]:
# Java: List<Integer> squares = numbers.stream()#           .map(x -> x * x)#           .collect(Collectors.toList());numbers = range(1, 11)squares = [x**2 for x in numbers]print(squares)# With filter# Java: .filter(x -> x % 2 == 0).map(x -> x * x)even_squares = [x**2 for x in numbers if x % 2 == 0]print(even_squares)# Nested comprehension (flatten)matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]flat = [val for row in matrix for val in row]print(flat)

In [ ]:
# Dict comprehensionwords = ["spark", "hadoop", "airflow", "python"]word_lengths = {w: len(w) for w in words}print(word_lengths)# Set comprehensionunique_lengths = {len(w) for w in words}print(unique_lengths)# 💡 Real-world: transform column namesraw_columns = ["First Name", "Last Name", "Date Of Birth"]clean = {col: col.lower().replace(" ", "_") for col in raw_columns}print(clean)

## Generators☕ **Java parallel:** Like `Stream.generate()` or `Iterator`, but with `yield`.Generators produce values lazily — critical for processing large datasets that don't fit in memory.

In [ ]:
# Generator function — uses yield instead of returndef read_chunks(file_path: str, chunk_size: int = 1024):    """Read a file in chunks — memory efficient."""    with open(file_path, 'r') as f:        while True:            chunk = f.read(chunk_size)            if not chunk:                break            yield chunk# Generator expression — like list comp but with ()# This does NOT load everything into memoryline_lengths = (len(line) for line in ["short", "medium length", "a very long string"])print(type(line_lengths))  # <class 'generator'>print(list(line_lengths))  # consumed on demand

In [ ]:
# 💡 Real-world: processing large data files row by rowimport csvfrom io import StringIOcsv_data = """id,name,amount1,Alice,100.502,Bob,200.753,Charlie,50.25"""def parse_transactions(data: str):    """Yield parsed transactions one at a time."""    reader = csv.DictReader(StringIO(data))    for row in reader:        yield {            "id": int(row["id"]),            "name": row["name"],            "amount": float(row["amount"])        }# Process without loading all into memorytotal = sum(t["amount"] for t in parse_transactions(csv_data))print(f"Total: ${total:.2f}")

## Decorators☕ **Java parallel:** Like annotations (`@Override`, `@Transactional`) but they actually wrap the function. Closer to AOP.

In [ ]:
import timefrom functools import wraps# A decorator is a function that wraps another functiondef timer(func):    """Measure execution time of a function."""    @wraps(func)  # preserves original function's name/docstring    def wrapper(*args, **kwargs):        start = time.perf_counter()        result = func(*args, **kwargs)        elapsed = time.perf_counter() - start        print(f"{func.__name__} took {elapsed:.4f}s")        return result    return wrapper@timerdef process_data(n: int) -> list:    """Simulate data processing."""    return [i**2 for i in range(n)]result = process_data(1_000_000)print(f"Processed {len(result)} records")

In [ ]:
# Decorator with parametersdef retry(max_attempts: int = 3, delay: float = 1.0):    """Retry a function on failure — useful for API calls, DB connections."""    def decorator(func):        @wraps(func)        def wrapper(*args, **kwargs):            for attempt in range(1, max_attempts + 1):                try:                    return func(*args, **kwargs)                except Exception as e:                    if attempt == max_attempts:                        raise                    print(f"Attempt {attempt} failed: {e}. Retrying...")                    time.sleep(delay)        return wrapper    return decorator@retry(max_attempts=3, delay=0.1)def fetch_data(url: str) -> str:    """Simulate an unreliable API call."""    import random    if random.random() < 0.6:        raise ConnectionError("API timeout")    return f"Data from {url}"try:    result = fetch_data("https://api.example.com/data")    print(result)except ConnectionError:    print("All retries exhausted")

## Context Managers☕ **Java parallel:** Like `AutoCloseable` + `try-with-resources`, but more flexible.

In [ ]:
from contextlib import contextmanagerimport time@contextmanagerdef pipeline_step(step_name: str):    """Track pipeline step execution."""    print(f"▶ Starting: {step_name}")    start = time.perf_counter()    try:        yield  # control returns to the 'with' block    except Exception as e:        print(f"✗ Failed: {step_name} — {e}")        raise    else:        elapsed = time.perf_counter() - start        print(f"✓ Completed: {step_name} ({elapsed:.3f}s)")# Usagewith pipeline_step("Extract"):    data = list(range(1000))with pipeline_step("Transform"):    data = [x * 2 for x in data]with pipeline_step("Load"):    print(f"  Loading {len(data)} records...")

## 🏋️ Exercise: Build a Pipeline Logger DecoratorCreate a decorator `@log_pipeline` that logs function name, arguments, result count, and duration.

In [ ]:
# Exercise: implement this decoratordef log_pipeline(func):    """Log pipeline step: function name, args, result length, and duration."""    # YOUR CODE HERE    pass@log_pipelinedef extract(source: str) -> list:    return [{"id": i, "source": source} for i in range(100)]@log_pipelinedef transform(records: list) -> list:    return [r for r in records if r["id"] % 2 == 0]data = extract("healthcare_db")filtered = transform(data)

In [ ]:
# Solutiondef log_pipeline(func):    @wraps(func)    def wrapper(*args, **kwargs):        start = time.perf_counter()        result = func(*args, **kwargs)        elapsed = time.perf_counter() - start        count = len(result) if hasattr(result, '__len__') else '?'        print(f"[{func.__name__}] args={args}, kwargs={kwargs}")        print(f"  → {count} records in {elapsed:.4f}s")        return result    return wrapper@log_pipelinedef extract(source: str) -> list:    return [{"id": i, "source": source} for i in range(100)]@log_pipelinedef transform(records: list) -> list:    return [r for r in records if r["id"] % 2 == 0]data = extract("healthcare_db")filtered = transform(data)